In [61]:
import pandas as pd
import numpy as np
import re
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

Load Datasets

In [62]:
df_enforcement = pd.read_excel("raw/enforcement_actions.xlsx")
df_facility_types = pd.read_excel("raw/lookup_facility_types.xlsx")
df_narratives = pd.read_csv("raw/ltc_narratives.csv", encoding="latin-1")

In [63]:
df_enforcement

,FACID,FACILITY_NAME,LTC,FAC_TYPE_CODE,FAC_FDR,DISTRICT_OFFICE,PENALTY_ISSUE_DATE,PENALTY_NUMBER,DISPOSITION,PENALTY_TYPE,...,TOTAL_AMOUNT_DUE_FINAL,TOTAL_PENALTY_OFFSET_AMOUNT,TOTAL_COLLECTED_AMOUNT,TOTAL_BALANCE_DUE,ACLAIMS_VISIT_NUMBER,EVENTID,DEATH_RELATED,INTAKEID_ALL,PRIORITY_ALL,SFY
0,10000060,"SEAVIEW REHABILITATION & WELLNESS CENTER, LP",LTC,SNF,Skilled Nursing Facility,Santa Rosa,2000-01-06,10001426,Closed,Citation,...,1000.0,-350.0,650.0,0.0,10010610.0,A99999,N,NaN,NaN,SFY1999-00
1,10000061,PALM DRIVE NURSING AND REHABILITATION CENTER,LTC,SNF,Skilled Nursing Facility,Santa Rosa,2000-01-26,10001435,Closed,Citation,...,7000.0,0.0,7000.0,0.0,10010652.0,A99999,N,NaN,NaN,SFY1999-00
2,10000043,PARK VIEW POST ACUTE,LTC,SNF,Skilled Nursing Facility,Santa Rosa,2000-01-26,10001436,Closed,Citation,...,1500.0,0.0,1500.0,0.0,10010654.0,A99999,N,NaN,NaN,SFY1999-00
3,10000043,PARK VIEW POST ACUTE,LTC,SNF,Skilled Nursing Facility,Santa Rosa,2000-01-27,10001437,Closed,Citation,...,1000.0,0.0,1000.0,0.0,10010657.0,A99999,N,NaN,NaN,SFY1999-00
4,10000062,"SHERWOOD OAKS POST ACUTE CARE, LLC",LTC,SNF,Skilled Nursing Facility,Santa Rosa,2000-01-28,10001438,Closed,Citation,...,1000.0,-350.0,650.0,0.0,10010659.0,A99999,N,NaN,NaN,SFY1999-00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20545,930000095,MEMORIALCARE LONG BEACH MEDICAL CENTER,Non-LTC,GACH,General Acute Care Hospital,LA Acute/Ancillary,2023-08-15,930019771,Open,Administrative Penalty,...,20000.0,0.0,0.0,20000.0,NaN,K5TO11,NaN,CA00810014,D,SFY2023-24
20546,930000129,VALLEY PRESBYTERIAN HOSPITAL,Non-LTC,GACH,General Acute Care Hospital,LA Acute/Ancillary,2023-10-24,930019974,Closed,Administrative Penalty,...,125000.0,-31250.0,93750.0,0.0,NaN,1CJ911,NaN,CA00777415,C,SFY2023-24
20547,930000129,VALLEY PRESBYTERIAN HOSPITAL,Non-LTC,GACH,General Acute Care Hospital,LA Acute/Ancillary,2023-04-11,930019977,Closed,Administrative Penalty,...,250000.0,-62500.0,187500.0,0.0,NaN,2TCN11,NaN,CA00799417,C,SFY2022-23
20548,930000133,PACIFICA HOSPITAL OF THE VALLEY,Non-LTC,GACH,General Acute Care Hospital,LA Acute/Ancillary,2024-04-24,930020061,Open,Administrative Penalty,...,3636.0,0.0,0.0,3636.0,NaN,DELK11,N,CA00880498,B,SFY2023-24


Death Related 

In [64]:
death_related=df_enforcement[df_enforcement['DEATH_RELATED']=='Y']
death_related['FAC_FDR'].value_counts()


FAC_FDR
Skilled Nursing Facility                    727
General Acute Care Hospital                 228
Intermediate Care Facility-DD/H/N/CN/IID     76
Intermediate Care Facility                   19
Acute Psychiatric Hospital                    9
Congregate Living Health Facility             7
Name: count, dtype: int64

In [65]:
death_related=df_enforcement[df_enforcement['DEATH_RELATED']=='N']
death_related['FAC_FDR'].value_counts()

FAC_FDR
Skilled Nursing Facility                        12618
Intermediate Care Facility-DD/H/N/CN/IID         2433
General Acute Care Hospital                      1760
Congregate Living Health Facility                 269
Intermediate Care Facility                        147
Acute Psychiatric Hospital                         24
Pediatric Day Health & Respite Care Facility        4
Surgical Clinic                                     1
Name: count, dtype: int64

In [66]:
death_related=df_enforcement[df_enforcement['DEATH_RELATED']=='Y']
death_related['DISTRICT_OFFICE'].value_counts()


DISTRICT_OFFICE
LA Region 2                 117
LA Region 3                  93
LA Region 1                  76
Sacramento                   72
Santa Rosa                   70
Riverside                    65
Orange                       61
San Diego                    57
East Bay                     55
San Jose                     53
State Facilities Section     52
San Bernardino               45
San Francisco                41
Chico                        40
LA Acute/Ancillary           40
Fresno                       34
Bakersfield                  34
Ventura                      24
Stockton                     21
LA ICF/DD/Clinics            11
LA HHA/Hospice                5
Name: count, dtype: int64

In [67]:
# Total citations per district
total = df_enforcement.groupby('DISTRICT_OFFICE').size()

# Death related per district
deaths = df_enforcement[df_enforcement['DEATH_RELATED'] == 'Y'].groupby('DISTRICT_OFFICE').size()

# Death rate per district
death_rate = (deaths / total * 100).round(2).sort_values(ascending=False)
print(death_rate)

DISTRICT_OFFICE
LA Acute/Ancillary          17.09
State Facilities Section     8.33
LA Region 2                  7.46
San Francisco                7.21
San Diego                    6.74
Riverside                    6.34
San Bernardino               6.02
Stockton                     5.75
Sacramento                   5.68
Orange                       5.57
LA Region 1                  5.24
East Bay                     5.13
Fresno                       4.88
LA Region 3                  4.88
Bakersfield                  4.07
Santa Rosa                   3.68
San Jose                     3.44
Ventura                      3.39
Chico                        3.31
LA HHA/Hospice               2.26
LA ICF/DD/Clinics            1.64
dtype: float64


In [68]:
# citations per facility per district
total_citations = df_enforcement.groupby('DISTRICT_OFFICE').size()
unique_facilities = df_enforcement.groupby('DISTRICT_OFFICE')['FACID'].nunique()

citations_per_facility = (total_citations / unique_facilities).round(2).sort_values(ascending=False)
print(citations_per_facility)

DISTRICT_OFFICE
State Facilities Section    17.83
LA Region 3                 15.37
LA Region 2                 11.21
LA Region 1                 10.89
San Jose                    10.70
Santa Rosa                  10.50
Chico                       10.50
Sacramento                   8.99
Stockton                     8.90
Bakersfield                  8.44
Riverside                    7.88
San Francisco                6.77
East Bay                     6.20
Ventura                      6.15
San Bernardino               5.49
Orange                       5.45
Fresno                       5.01
San Diego                    4.62
LA Acute/Ancillary           3.12
LA HHA/Hospice               2.83
LA ICF/DD/Clinics            2.76
dtype: float64


In [69]:
df_narratives['NARRATIVE_CLEAN'] = df_narratives['NARRATIVE'].str.replace(r'\b\d+\b', '', regex=True)

# Fit TF-IDF
tfidf = TfidfVectorizer(
    max_features=100,
    stop_words='english',
    ngram_range=(1,2),
    token_pattern=r'[a-zA-Z]{3,}'  # only words 3+ letters, no numbers
)

matrix = tfidf.fit_transform(df_narratives['NARRATIVE_CLEAN'].fillna(''))
terms = tfidf.get_feature_names_out()

# Get top 3 keywords per citation
def get_top_keywords(row, terms, n=3):
    top_indices = np.argsort(row)[-n:][::-1]
    return ', '.join([terms[i] for i in top_indices])

df_narratives['TOP_KEYWORDS'] = [
    get_top_keywords(matrix[i].toarray()[0], terms) 
    for i in range(matrix.shape[0])
]

print(df_narratives[['PENALTY_NUMBER', 'TOP_KEYWORDS']].head(10))

   PENALTY_NUMBER                 TOP_KEYWORDS
0        20008964    resident, according, skin
1        20009068  patient, hospital, facility
2        20009069      resident, facility, bed
3        20009078    resident, blood, pressure
4        20009082    resident, condition, pain
5        20009084   resident, stated, facility
6        20009140   patient, facility, nursing
7        20009162    abuse, resident, incident
8        20009163    facility, abuse, resident
9        20009260           resident, cna, don


In [ ]:
df_narratives['TOP_KEYWORDS']

In [ ]:
df_enforcement.describe()

In [ ]:
df_facility_types.describe()

In [ ]:
df_narratives.describe()

In [ ]:
df_narratives.isnull().sum()

In [70]:
os.makedirs("cleaned", exist_ok=True)

def clean_text(text):
    if pd.isna(text):
        return ''
    text = re.sub(r'F\d{3,4}', '', text)          
    text = re.sub(r'T\d{2}', '', text)             
    text = re.sub(r'DIV\d+', '', text)             
    text = re.sub(r'CH\d+', '', text)             
    text = re.sub(r'ART\d+', '', text)             
    text = re.sub(r'\([^)]*\)', '', text)         
    text = re.sub(r'\b\d+\b', '', text)           
    text = re.sub(r'[^\w\s]', ' ', text)           
    text = re.sub(r'\n+', ' ', text)              
    text = re.sub(r'\s+', ' ', text)              
    return text.strip().lower()

print("Cleaning enforcement actions...")
df_enforcement.columns = df_enforcement.columns.str.strip().str.upper()
df_enforcement['PENALTY_NUMBER']    = df_enforcement['PENALTY_NUMBER'].astype(str).str.strip()
df_enforcement['FAC_TYPE_CODE']     = df_enforcement['FAC_TYPE_CODE'].astype(str).str.strip().str.upper()
df_enforcement['FACID']             = df_enforcement['FACID'].astype(str).str.strip()
df_enforcement['PENALTY_ISSUE_DATE']= pd.to_datetime(df_enforcement['PENALTY_ISSUE_DATE'], errors='coerce')
df_enforcement['VIOLATION_FROM_DATE']= pd.to_datetime(df_enforcement['VIOLATION_FROM_DATE'], errors='coerce')
df_enforcement['DEATH_RELATED']     = df_enforcement['DEATH_RELATED'].fillna('N')


print("Cleaning narratives...")
df_narratives.columns = df_narratives.columns.str.strip().str.upper()
df_narratives['PENALTY_NUMBER'] = df_narratives['PENALTY_NUMBER'].astype(str).str.strip()
df_narratives['FACID']          = df_narratives['FACID'].astype(str).str.strip()
df_narratives['NARRATIVE_CLEAN'] = df_narratives['NARRATIVE'].apply(clean_text)


print("Extracting TF-IDF keywords...")
custom_stop_words = list(ENGLISH_STOP_WORDS) + [
    'resident', 'facility', 'patient', 'stated', 'according',
    'don', 'staff', 'nursing', 'hospital', 'following',
    'included', 'indicated', 'required', 'titled', 'dated',
    'review', 'record', 'noted', 'note', 'shall', 'use',
    'used', 'did', 'day', 'days', 'time', 'right', 'left'
]

tfidf = TfidfVectorizer(
    max_features=100,
    stop_words=custom_stop_words,
    ngram_range=(1, 2),
    token_pattern=r'[a-zA-Z]{3,}'
)

matrix = tfidf.fit_transform(df_narratives['NARRATIVE_CLEAN'].fillna(''))
terms  = tfidf.get_feature_names_out()

def get_top_keywords(row, terms, n=3):
    top_indices = np.argsort(row)[-n:][::-1]
    return ', '.join([terms[i] for i in top_indices])

df_narratives['TOP_KEYWORDS'] = [
    get_top_keywords(matrix[i].toarray()[0], terms)
    for i in range(matrix.shape[0])
]
df_narratives = df_narratives.drop(columns=['NARRATIVE'])

print("Cleaning lookup table...")
df_facility_types.columns = df_facility_types.columns.str.strip().str.upper()
df_facility_types['VALUE'] = df_facility_types['VALUE'].astype(str).str.strip().str.upper()
df_facility_types = df_facility_types.dropna(subset=['VALUE'])


print("Saving cleaned files...")
df_enforcement.to_csv("cleaned/enforcement_actions.csv", index=False)
df_narratives.to_csv("cleaned/ltc_narratives.csv", index=False)
df_facility_types.to_csv("cleaned/lookup_facility_types.csv", index=False)

print(f"\nDone!")
print(f"  Enforcement actions: {df_enforcement.shape}")
print(f"  Narratives:          {df_narratives.shape}")
print(f"  Lookup:              {df_facility_types.shape}")
print(f"\nCleaned files saved to cleaned/")

Cleaning enforcement actions...
Cleaning narratives...
Extracting TF-IDF keywords...
Cleaning lookup table...
Saving cleaned files...

Done!
  Enforcement actions: (20550, 31)
  Narratives:          (2885, 9)
  Lookup:              (192, 4)

Cleaned files saved to cleaned/


In [71]:
df_narrative_cleaned= pd.read_csv("cleaned/ltc_narratives.csv")
df_narrative_cleaned

,FACID,FACILITY_NAME,PENALTY_NUMBER,CLASS_ASSESSED_INITIAL,PENALTY_ISSUE_DATE,EVENTID,NARRATIVE LENGTH,NARRATIVE_CLEAN,TOP_KEYWORDS
0,20000132,Fremont HealthCare Center,20008964,B,01-Feb-12,V6XV11,4029,free of accident hazards supervision devices t...,"skin, supervision, prevent"
1,20000054,Willow Tree Nursing Center,20009068,B,02-Mar-12,K04O11,2678,title if a patient of a skilled nursing facili...,"discharge, bed, showed"
2,140000051,Kindred Nursing and Rehabilitation - Ygnacio V...,20009069,B,02-Mar-12,FGDY11,7868,permitting resident to return to facility a nu...,"bed, medical, administrator"
3,20000026,"Bay View Rehabilitation Hospital, LLC",20009078,AA,05-Mar-12,8JFW11,9084,provide care services for highest well beingea...,"blood, pressure, showed"
4,140000105,Lone Tree Convalescent Hospital,20009082,B,06-Mar-12,PUKB11,5560,nursing service general nursing service shall ...,"wound, condition, assessment"
...,...,...,...,...,...,...,...,...,...
2880,960000984,"HARBOR HEALTH CARE, INC. - BAYLOR DIVISION",960013663,B,1-Dec-17,D3EI11,7221,california code of regulations title there sha...,"client, services, treatment"
2881,960000984,"HARBOR HEALTH CARE, INC. - BAYLOR DIVISION",960013659,B,1-Dec-17,D3EI11,6314,california code of regulations title there sha...,"client, services, treatment"
2882,630014895,Veterans Home of California - Redding,150013628,A,21-Nov-17,2UBD11,10816,patient rights cfr prohibit mistreatment negle...,"shift, wound, cna"
2883,630013324,Citrus Home ICF/DD-N,30013591,B,1-Nov-17,26RO11,2651,health and safety code a long term health care...,"client, abuse, department"


In [ ]:
narratives_keys = set(df_narratives['PENALTY_NUMBER'])
enforcement_keys = set(df_enforcement['PENALTY_NUMBER'])
print(f"Matching keys: {len(narratives_keys & enforcement_keys)}")
print(f"Narratives only: {len(narratives_keys - enforcement_keys)}")
print(f"Enforcement only: {len(enforcement_keys - narratives_keys)}")